# Ethiopian Medical Telegram — Exploratory Data Analysis
Run this notebook **after** `dbt run` has built the mart tables.

Sections:
1. Setup & connection
2. Channel overview
3. Posting volume trends
4. Top product terms (word frequency)
5. Engagement analysis (views & forwards)
6. Image content breakdown (YOLO categories)
7. Day-of-week patterns

In [1]:
import os
import warnings
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sqlalchemy
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv('../.env')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

DB_URL = (
    f"postgresql://{os.getenv('POSTGRES_USER','warehouse_user')}"
    f":{os.getenv('POSTGRES_PASSWORD','warehouse_pass')}"
    f"@{os.getenv('POSTGRES_HOST','localhost')}"
    f":{os.getenv('POSTGRES_PORT','5432')}"
    f"/{os.getenv('POSTGRES_DB','medical_warehouse')}"
)

engine = sqlalchemy.create_engine(DB_URL)

def q(sql):
    return pd.read_sql(sql, engine)

print('Connected to:', DB_URL.split('@')[1])

ModuleNotFoundError: No module named 'psycopg2'

## 1. Channel Overview

In [ ]:
channels = q("""
    SELECT channel_name, channel_type, total_posts, avg_views,
           first_post_date, last_post_date
    FROM marts.dim_channels
    ORDER BY total_posts DESC
""")
channels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(channels['channel_name'], channels['total_posts'], color='steelblue')
axes[0].set_xlabel('Total Posts')
axes[0].set_title('Posts per Channel')

axes[1].barh(channels['channel_name'], channels['avg_views'], color='coral')
axes[1].set_xlabel('Average Views')
axes[1].set_title('Average Views per Channel')

plt.tight_layout()
plt.savefig('channel_overview.png', dpi=150)
plt.show()

## 2. Daily Posting Volume (last 90 days)

In [ ]:
daily = q("""
    SELECT
        dd.full_date       AS date,
        dc.channel_name,
        COUNT(*)           AS posts,
        SUM(view_count)    AS total_views
    FROM marts.fct_messages fm
    JOIN marts.dim_channels dc ON fm.channel_key = dc.channel_key
    JOIN marts.dim_dates    dd ON fm.date_key     = dd.date_key
    WHERE dd.full_date >= CURRENT_DATE - INTERVAL '90 days'
    GROUP BY dd.full_date, dc.channel_name
    ORDER BY dd.full_date
""")
daily['date'] = pd.to_datetime(daily['date'])

pivot = daily.pivot_table(index='date', columns='channel_name', values='posts', fill_value=0)

pivot.plot(kind='area', alpha=0.6, figsize=(14, 5))
plt.title('Daily Posting Volume by Channel (last 90 days)')
plt.xlabel('')
plt.ylabel('Messages per Day')
plt.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('daily_volume.png', dpi=150)
plt.show()

## 3. Weekly Trend (rolling 7-day average)

In [ ]:
weekly = daily.groupby('date')['posts'].sum().reset_index()
weekly['rolling_7'] = weekly['posts'].rolling(7).mean()

plt.figure(figsize=(14, 4))
plt.bar(weekly['date'], weekly['posts'], alpha=0.4, color='steelblue', label='Daily posts')
plt.plot(weekly['date'], weekly['rolling_7'], color='darkblue', linewidth=2, label='7-day avg')
plt.title('Overall Daily Posting Volume — All Channels')
plt.ylabel('Messages')
plt.legend()
plt.tight_layout()
plt.savefig('weekly_trend.png', dpi=150)
plt.show()

## 4. Top 20 Most Mentioned Product Terms

In [ ]:
texts = q("SELECT message_text FROM marts.fct_messages WHERE message_text <> ''")

STOPWORDS = {
    'the','and','for','with','from','that','this','have','will',
    'your','been','they','were','also','into','more','than','some',
    'when','then','over','each','just','are','not','has','its',
    'can','now','new','per','our','all','was','you','get','use',
    'etb','mg','ml','tab','tabs','pack','box','available','contact',
}

counter = Counter()
for text in texts['message_text']:
    words = re.findall(r'[a-z]{4,}', text.lower())
    counter.update(w for w in words if w not in STOPWORDS)

top_terms = pd.DataFrame(counter.most_common(20), columns=['term', 'count'])

plt.figure(figsize=(12, 6))
plt.barh(top_terms['term'][::-1], top_terms['count'][::-1], color='mediumseagreen')
plt.xlabel('Mentions')
plt.title('Top 20 Most Mentioned Product Terms Across All Channels')
plt.tight_layout()
plt.savefig('top_terms.png', dpi=150)
plt.show()

top_terms

## 5. Engagement Analysis — Views Distribution

In [ ]:
engagement = q("""
    SELECT
        dc.channel_name,
        fm.view_count,
        fm.forward_count,
        fm.has_image
    FROM marts.fct_messages fm
    JOIN marts.dim_channels dc ON fm.channel_key = dc.channel_key
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ch, grp in engagement.groupby('channel_name'):
    axes[0].hist(grp['view_count'].clip(upper=5000), bins=40, alpha=0.5, label=ch)
axes[0].set_xlabel('Views (capped at 5 000)')
axes[0].set_title('View Count Distribution by Channel')
axes[0].legend(fontsize=8)

img_views = engagement.groupby('has_image')['view_count'].mean().rename({True:'With Image', False:'Text Only'})
img_views.plot(kind='bar', ax=axes[1], color=['coral','steelblue'], rot=0)
axes[1].set_title('Average Views: Image Posts vs Text-Only Posts')
axes[1].set_ylabel('Average Views')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('engagement.png', dpi=150)
plt.show()

print("\nAverage views by image presence:")
print(engagement.groupby('has_image')['view_count'].describe().round(1))

## 6. YOLO Image Category Breakdown

In [ ]:
yolo = q("""
    SELECT
        dc.channel_name,
        fid.image_category,
        COUNT(*)              AS count,
        AVG(fid.view_count)   AS avg_views
    FROM marts.fct_image_detections fid
    JOIN marts.dim_channels dc ON fid.channel_key = dc.channel_key
    GROUP BY dc.channel_name, fid.image_category
    ORDER BY dc.channel_name, count DESC
""")

if yolo.empty:
    print("No YOLO data yet — run src/yolo_detect.py or scripts/generate_sample_yolo.py first.")
else:
    pivot_yolo = yolo.pivot_table(
        index='channel_name', columns='image_category',
        values='count', fill_value=0
    )
    pivot_yolo.plot(kind='bar', figsize=(12, 5), rot=20)
    plt.title('Image Category Distribution by Channel (YOLO)')
    plt.ylabel('Number of Images')
    plt.xlabel('')
    plt.legend(title='Category')
    plt.tight_layout()
    plt.savefig('yolo_categories.png', dpi=150)
    plt.show()

    print("\nDo promotional posts get more views?")
    views_by_cat = yolo.groupby('image_category')['avg_views'].mean().sort_values(ascending=False)
    print(views_by_cat.round(0))

## 7. Day-of-Week Posting Patterns

In [ ]:
dow = q("""
    SELECT
        dd.day_name,
        dd.day_of_week,
        COUNT(*)         AS posts,
        AVG(view_count)  AS avg_views
    FROM marts.fct_messages fm
    JOIN marts.dim_dates dd ON fm.date_key = dd.date_key
    GROUP BY dd.day_name, dd.day_of_week
    ORDER BY dd.day_of_week
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(dow['day_name'].str.strip(), dow['posts'], color='mediumpurple')
axes[0].set_title('Total Posts by Day of Week')
axes[0].set_ylabel('Post Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].plot(dow['day_name'].str.strip(), dow['avg_views'], marker='o', color='tomato', linewidth=2)
axes[1].set_title('Average Views by Day of Week')
axes[1].set_ylabel('Avg Views')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('day_of_week.png', dpi=150)
plt.show()

dow[['day_name','posts','avg_views']].assign(
    day_name=dow['day_name'].str.strip(),
    avg_views=dow['avg_views'].round(0)
)

## 8. Price Extraction — Products Mentioning ETB

In [ ]:
priced = q("""
    SELECT
        dc.channel_name,
        fm.message_text,
        fm.view_count
    FROM marts.fct_messages fm
    JOIN marts.dim_channels dc ON fm.channel_key = dc.channel_key
    WHERE LOWER(fm.message_text) LIKE '%etb%'
""")

def extract_prices(text):
    return [float(p) for p in re.findall(r'(\d+(?:\.\d+)?)\s*etb', text.lower())]

priced['prices'] = priced['message_text'].apply(extract_prices)
priced_flat = priced.explode('prices').dropna(subset=['prices'])
priced_flat['prices'] = priced_flat['prices'].astype(float)
priced_clean = priced_flat[priced_flat['prices'].between(5, 5000)]

print(f"Messages with ETB prices: {len(priced)}")
print(f"Extracted price points   : {len(priced_clean)}")
print("\nPrice stats per channel:")
print(priced_clean.groupby('channel_name')['prices'].describe().round(1))

priced_clean.boxplot(column='prices', by='channel_name', figsize=(12, 5), rot=20)
plt.suptitle('')
plt.title('Price Distribution by Channel (ETB)')
plt.ylabel('Price (ETB)')
plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150)
plt.show()

## 9. Summary — Key Insights

In [ ]:
total_msgs  = q("SELECT COUNT(*) AS n FROM marts.fct_messages").iloc[0]['n']
total_img   = q("SELECT COUNT(*) AS n FROM marts.fct_messages WHERE has_image").iloc[0]['n']
total_chan   = q("SELECT COUNT(*) AS n FROM marts.dim_channels").iloc[0]['n']
avg_views_all = q("SELECT AVG(view_count) AS v FROM marts.fct_messages").iloc[0]['v']

print("="*45)
print("  ETHIOPIAN MEDICAL TELEGRAM — SUMMARY")
print("="*45)
print(f"  Total channels analysed : {total_chan}")
print(f"  Total messages          : {total_msgs:,}")
print(f"  Messages with images    : {total_img:,} ({100*total_img/max(total_msgs,1):.1f}%)")
print(f"  Average views / message : {avg_views_all:,.0f}")
print("="*45)